In [2]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [9]:
!pip install 'stable-baselines3[extra]'
!pip install 'gymnasium[box2d]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 100.2 MB/s eta 0:00:00


In [10]:
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor


In [11]:
env = gym.make("LunarLander-v3", continuous=False, gravity=-10.0,
               enable_wind=False, wind_power=15.0, turbulence_power=1.5)
observation, info = env.reset()

for _ in range(20):
  action = env.action_space.sample()
  print("Action taken:", action)
  observation, reward, terminated, truncated, info = env.step(action)
  if terminated or truncated:
      print("Environment is reset")
      observation, info = env.reset()

env.close()


Action taken: 1
Action taken: 0
Action taken: 3
Action taken: 2
Action taken: 0
Action taken: 0
Action taken: 2
Action taken: 1
Action taken: 1
Action taken: 1
Action taken: 1
Action taken: 3
Action taken: 3
Action taken: 0
Action taken: 0
Action taken: 1
Action taken: 2
Action taken: 3
Action taken: 3
Action taken: 2


<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


In [12]:
env.reset()
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample())

_____OBSERVATION SPACE_____ 

Observation Space Shape (8,)
Sample observation [ 1.6813434   1.5893462  -0.23949957  7.450587    4.1814256   1.3081018
  0.68506765  0.89954257]


In [13]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample())


 _____ACTION SPACE_____ 

Action Space Shape 4
Action Space Sample 3


In [15]:
env = gym.make("LunarLander-v3", continuous=False)
env.reset()

(array([ 0.0071434 ,  1.4026129 ,  0.723521  , -0.36922514, -0.0082705 ,
        -0.16388817,  0.        ,  0.        ], dtype=float32),
 {})

In [18]:
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1
)
model.learn(total_timesteps=1_000_000)
model.save("ppo_lunar")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Streaming output truncated to the last 5000 lines.
|    value_loss           | 14.1         |
------------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 467         |
|    ep_rew_mean          | 229         |
| time/                   |             |
|    fps                  | 532         |
|    iterations           | 740         |
|    time_elapsed         | 1422        |
|    total_timesteps      | 757760      |
| train/                  |             |
|    approx_kl            | 0.005033505 |
|    clip_fraction        | 0.0305      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.807      |
|    explained_variance   | 0.952       |
|    learning_rate        | 0.0003      |
|    loss                 | 15.4        |
|    n_updates            | 2956        |
|    policy_gradient_loss | -0.00618    |
|    value_loss           | 23          |
-----------------------

In [19]:
eval_env = Monitor(gym.make("LunarLander-v3", continuous=False))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=259.04 +/- 16.817645186821334
